# PROYECTO PTIA 2026-2
## StudyFlow AI: capa de validación

**Integrantes:**

Karol Ximena Rodriguez Reyes
Juan David Moreno D'Aleman

---

### ¿Qué vamos a hacer en este notebook?

`LLM (entiende lo que pide el estudiante)` -> **`Validador (¿cabe en el tiempo?)`** -> `MLP (prioriza y reparte)` -> `Capa de asignación (arma las sesiones)`

Aquí desarrollamos **solo el validador**, en tres piezas, y lo probamos antes de pasarlo a archivos `.py`:

1. **Pydantic (`models.py`):** ¿los datos tienen forma y valores válidos?
2. **Capacidad efectiva (`disponibilidad.py`):** ¿cuántas horas reales de estudio ofrece cada día? También la usará el MLP (entradas 5 y 6).
3. **Coherencia (`validador.py`):** ¿el plan es factible?

Flujo: `entrada -> Pydantic -> validador de factibilidad -> características -> MLP`.

Todo el código, a excepción de las pruebas, se mudará a archivos `.py` para conectar los módulos entre sí y el front con el back.

# PASO 1. REGLAS Y DECISIONES

## 1.1 Qué revisa cada capa

| Capa | Pregunta | Ejemplos |
|---|---|---|
| Pydantic | ¿Los datos son válidos? | nombre vacío, 0 horas, franja que termina antes de empezar, franjas traslapadas, días repetidos |
| Coherencia | ¿El plan es posible? | más horas de trabajo que tiempo disponible, plazos demasiado cortos |

Pydantic no depende de ningún otro módulo. El validador importa los modelos, nunca al revés.

## 1.2 Capacidad efectiva de un día

La capacidad **no** es la suma bruta de las franjas:

* Cada franja se redondea hacia abajo a **bloques de 30 min** (`BLOQUE_MIN`). Una franja de 50 min aporta 30 min.
* `duracion_sesion` (30 a 240 min) es el **máximo** por sesión, y 30 min es el mínimo. Así no se pierde el sobrante: en 4 h con sesión máxima de 1 h 30 caben 1 h 30 + 1 h 30 + 1 h.
* Si hay `max_horas_dia`, la capacidad del día no lo supera.
* **El día de la entrega no cuenta** como día de trabajo. Esto coincide con el dataset del MLP: una actividad con `dias_restantes = d` tiene `d` días de trabajo.

## 1.3 Reglas del validador

| # | Qué revisa | Resultado |
|---|---|---|
| R1 | **Factibilidad acumulada:** para cada fecha límite *d*, las horas que vencen hasta *d* deben caber en la capacidad de los días anteriores a *d* (+ `MARGEN_HORAS`) | Error |
| R2 | **Ventana mínima:** al menos `VENTANA_MINIMA_DIAS` (3) días con disponibilidad antes de la entrega, porque el reparto tiene 3 tramos | Error si `modo="nuevo"`, advertencia si `modo="replanificacion"` |
| R3a | `duracion_sesion` mayor que `max_horas_dia` | Error |
| R3b | Día con franjas pero sin capacidad (todas duran menos de 30 min) | Advertencia por día |
| R4 | Fecha límite anterior al inicio del plan | Error |
| R5 | Disponibilidad de días anteriores al inicio del plan | Advertencia (se ignora) |
| R6 | Nombres de actividad duplicados | Advertencia |
| R7 | Capacidad cero hasta una fecha límite | Queda cubierta por R1 |

Decisiones de uso:

* `fecha_inicio` es un **parámetro**, no se usa `date.today()` dentro del validador. Las pruebas son reproducibles y se puede planear otro rango (por ejemplo, la semana siguiente).
* `MARGEN_HORAS = 0`: un plan que usa el 100 % de la capacidad es válido. No hay advertencia por uso alto.
* Resultado: `estado` (`factible`, `factible_con_advertencias`, `inviable`), `errores` y `advertencias`. Cada elemento lleva `codigo`, `mensaje` y `datos`.
* **Sugerencia de recorte** (en R1): entre las actividades que vencen hasta la fecha crítica, se propone recortar primero las de prioridad baja y, a igualdad, la que tenga más horas, hasta cubrir el déficit.

## PASO 2. CAPA PYDANTIC (`models.py`)

| Modelo | Reglas |
|---|---|
| `Actividad` | `nombre` de 1 a 100 caracteres (se quitan espacios en los extremos); `prioridad` y `dificultad` en `Nivel` (acepta mayúsculas); `fecha_limite` es una fecha; `horas_estimadas` mayor que 0 y hasta 100 |
| `Franja` | `fin` posterior a `inicio` |
| `DisponibilidadDia` | las franjas del día no se traslapan |
| `PlanEntrada` | al menos una actividad y un día de disponibilidad; sin días repetidos; `max_horas_dia` opcional (0 a 8 h); `duracion_sesion` en 30, 60, ..., 240 min |

La fecha límite **no** se compara con "hoy" aquí: eso es R4 del validador.

In [ ]:
"""Modelos de entrada (forma y valores). Ver StudyFlow_CapaValidacion.ipynb, paso 2."""
from datetime import date, time
from enum import Enum, IntEnum

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator


class Nivel(str, Enum):
    BAJA = "baja"
    MEDIA = "media"
    ALTA = "alta"


class DuracionSesion(IntEnum):
    M30 = 30
    M60 = 60
    M90 = 90
    M120 = 120
    M150 = 150
    M180 = 180
    M210 = 210
    M240 = 240


class Actividad(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    nombre: str = Field(min_length=1, max_length=100)
    prioridad: Nivel
    dificultad: Nivel
    fecha_limite: date
    horas_estimadas: float = Field(gt=0, le=100)

    @field_validator("prioridad", "dificultad", mode="before")
    @classmethod
    def _nivel_sin_mayusculas(cls, v):
        return v.strip().lower() if isinstance(v, str) else v


class Franja(BaseModel):
    inicio: time
    fin: time

    @model_validator(mode="after")
    def _fin_despues_de_inicio(self):
        if self.fin <= self.inicio:
            raise ValueError("la franja debe terminar después de empezar")
        return self


class DisponibilidadDia(BaseModel):
    dia: date
    franjas: list[Franja]

    @model_validator(mode="after")
    def _sin_traslapes(self):
        ordenadas = sorted(self.franjas, key=lambda f: f.inicio)
        for a, b in zip(ordenadas, ordenadas[1:]):
            if b.inicio < a.fin:
                raise ValueError(f"las franjas {a.inicio}-{a.fin} y {b.inicio}-{b.fin} se traslapan")
        return self


class PlanEntrada(BaseModel):
    actividades: list[Actividad] = Field(min_length=1)
    disponibilidad: list[DisponibilidadDia] = Field(min_length=1)
    max_horas_dia: float | None = Field(default=None, gt=0, le=8)
    duracion_sesion: DuracionSesion

    @model_validator(mode="after")
    def _sin_dias_repetidos(self):
        dias = [d.dia for d in self.disponibilidad]
        if len(dias) != len(set(dias)):
            raise ValueError("hay días repetidos en la disponibilidad")
        return self

## 2.1 Pruebas de la capa Pydantic

Cada línea `rechaza(...)` comprueba que el dato inválido se rechace con el error esperado; las demás comprueban que lo válido se acepte y se normalice.

In [2]:
from pydantic import ValidationError


def rechaza(fn, fragmento):
    try:
        fn()
    except ValidationError as e:
        assert fragmento in str(e), str(e)
        return
    raise AssertionError(f"debía rechazar ({fragmento})")


act_ok = dict(nombre="  Parcial  ", prioridad="ALTA", dificultad=" Media", fecha_limite="2026-11-02", horas_estimadas=4)
a = Actividad(**act_ok)
assert a.nombre == "Parcial" and a.prioridad is Nivel.ALTA and a.dificultad is Nivel.MEDIA

rechaza(lambda: Actividad(**{**act_ok, "nombre": "   "}), "string_too_short")
rechaza(lambda: Actividad(**{**act_ok, "nombre": "x" * 101}), "string_too_long")
rechaza(lambda: Actividad(**{**act_ok, "horas_estimadas": 0}), "greater_than")
rechaza(lambda: Actividad(**{**act_ok, "horas_estimadas": 101}), "less_than_equal")
rechaza(lambda: Actividad(**{**act_ok, "prioridad": "urgente"}), "enum")
rechaza(lambda: Actividad(**{**act_ok, "fecha_limite": "mañana"}), "date")

Franja(inicio="18:00", fin="20:00")
rechaza(lambda: Franja(inicio="20:00", fin="18:00"), "terminar")
rechaza(lambda: Franja(inicio="18:00", fin="18:00"), "terminar")

franjas = lambda *pares: [{"inicio": i, "fin": f} for i, f in pares]
DisponibilidadDia(dia="2026-10-26", franjas=franjas(("18:00", "20:00"), ("20:00", "21:00")))  # pegadas, sin traslape
DisponibilidadDia(dia="2026-10-26", franjas=[])
rechaza(lambda: DisponibilidadDia(dia="2026-10-26", franjas=franjas(("18:00", "20:00"), ("19:00", "21:00"))), "traslapan")

base = dict(actividades=[act_ok], disponibilidad=[{"dia": "2026-10-26", "franjas": franjas(("18:00", "20:00"))}], duracion_sesion=60)
PlanEntrada(**base)
rechaza(lambda: PlanEntrada(**{**base, "actividades": []}), "too_short")
rechaza(lambda: PlanEntrada(**{**base, "disponibilidad": []}), "too_short")
rechaza(lambda: PlanEntrada(**{**base, "duracion_sesion": 45}), "enum")
rechaza(lambda: PlanEntrada(**{**base, "max_horas_dia": 0}), "greater_than")
rechaza(lambda: PlanEntrada(**{**base, "max_horas_dia": 9}), "less_than_equal")
rechaza(lambda: PlanEntrada(**{**base, "disponibilidad": base["disponibilidad"] * 2}), "repetidos")
assert PlanEntrada(**{**base, "duracion_sesion": 240, "max_horas_dia": 8}).duracion_sesion == 240

print("Capa Pydantic: todas las pruebas pasaron")

Capa Pydantic: todas las pruebas pasaron


# PASO 3. CAPACIDAD EFECTIVA (`disponibilidad.py`)

Convierte las franjas en **horas realmente utilizables**. Lo usan dos piezas: el validador (R1, R2, R3b) y, más adelante, el MLP para calcular `horas_disponibles_ventana` y `carga_relativa`. Por eso vive en su propio módulo y las dos usan exactamente el mismo cálculo.

Ejemplo: franjas 11:00-14:00 y 16:00-17:30 dan 3 h + 1 h 30 = **4.5 h**. Con `max_horas_dia = 4` el día aporta 4 h.

In [ ]:
"""Capacidad efectiva de estudio por día. Ver StudyFlow_CapaValidacion.ipynb, paso 3."""
from datetime import date, time

BLOQUE_MIN = 30  # unidad de capacidad y duración mínima de una sesión (minutos)


def _minutos(t: time) -> int:
    return t.hour * 60 + t.minute


def capacidad_dia(dia: DisponibilidadDia, max_horas_dia: float | None = None):
    minutos = sum((_minutos(f.fin) - _minutos(f.inicio)) // BLOQUE_MIN * BLOQUE_MIN for f in dia.franjas)
    if max_horas_dia is not None:
        minutos = min(minutos, int(max_horas_dia * 60) // BLOQUE_MIN * BLOQUE_MIN)
    return minutos / 60


def capacidad_entre(disponibilidad: list[DisponibilidadDia], max_horas_dia: float | None, inicio: date, fin: date):
    """Horas efectivas de los días d con inicio <= d < fin (el día `fin` no cuenta)."""
    return sum(capacidad_dia(d, max_horas_dia) for d in disponibilidad if inicio <= d.dia < fin)


def dias_con_capacidad(disponibilidad: list[DisponibilidadDia], max_horas_dia: float | None, inicio: date, fin: date):
    return sum(1 for d in disponibilidad if inicio <= d.dia < fin and capacidad_dia(d, max_horas_dia) > 0)

## 3.1 Pruebas de la capacidad

In [ ]:
from datetime import date


def dia(fecha, *pares):
    return DisponibilidadDia(dia=fecha, franjas=franjas(*pares))


d = dia("2026-10-26", ("11:00", "14:00"), ("16:00", "17:30"))
assert capacidad_dia(d) == 4.5
assert capacidad_dia(d, max_horas_dia=4) == 4.0
assert capacidad_dia(d, max_horas_dia=2.75) == 2.5          # el tope también se redondea a bloques de 30 min
assert capacidad_dia(dia("2026-10-26", ("18:00", "19:30"))) == 1.5     # 1 h 30 completa, sin perder el sobrante
assert capacidad_dia(dia("2026-10-26", ("16:00", "20:00"))) == 4.0
assert capacidad_dia(dia("2026-10-26", ("18:00", "18:50"))) == 0.5     # 50 min -> 30 min
assert capacidad_dia(dia("2026-10-26", ("18:00", "18:45"), ("19:00", "19:45"))) == 1.0   # cada franja se redondea sola
assert capacidad_dia(dia("2026-10-26", ("18:00", "18:20"))) == 0
assert capacidad_dia(dia("2026-10-26")) == 0

tres_dias = [dia(f"2026-10-{n}", ("18:00", "21:00")) for n in (26, 27, 28)]
assert capacidad_entre(tres_dias, None, date(2026, 10, 26), date(2026, 10, 28)) == 6   # el día 28 (fin) no cuenta
assert capacidad_entre(tres_dias, None, date(2026, 10, 27), date(2026, 10, 29)) == 6   # el día 26 queda antes del inicio
assert capacidad_entre(tres_dias, 1, date(2026, 10, 26), date(2026, 10, 29)) == 3
assert dias_con_capacidad(tres_dias + [dia("2026-10-29", ("18:00", "18:10"))], None, date(2026, 10, 26), date(2026, 10, 30)) == 3

print("Capacidad efectiva: todas las pruebas pasaron")

# PASO 4. VALIDADOR DE COHERENCIA (`validador.py`)

`validar(plan, fecha_inicio, modo)` aplica las reglas R1 a R6 y devuelve un `Resultado`. `modo` es `"nuevo"` (actividades que el usuario ingresa) o `"replanificacion"`; solo cambia la gravedad de R2.

Las actividades con fecha límite anterior al inicio se reportan en R4 y se excluyen de R1 y R2, para no dar errores falsos en cadena.

In [5]:
"""Validación de coherencia del plan (R1-R7). Ver StudyFlow_CapaValidacion.ipynb, paso 4."""
from collections import Counter
from datetime import date
from enum import Enum
from typing import Any, Literal

from pydantic import BaseModel, Field

MARGEN_HORAS = 0.0
VENTANA_MINIMA_DIAS = 3
_ORDEN_PRIORIDAD = {Nivel.BAJA: 0, Nivel.MEDIA: 1, Nivel.ALTA: 2}


class Estado(str, Enum):
    FACTIBLE = "factible"
    CON_ADVERTENCIAS = "factible_con_advertencias"
    INVIABLE = "inviable"


class Hallazgo(BaseModel):
    codigo: str
    mensaje: str
    datos: dict[str, Any] = Field(default_factory=dict)


class Resultado(BaseModel):
    estado: Estado
    errores: list[Hallazgo]
    advertencias: list[Hallazgo]


def _sugerir_recorte(vencen, deficit):
    candidatas = sorted(vencen, key=lambda a: (_ORDEN_PRIORIDAD[a.prioridad], -a.horas_estimadas))
    elegidas, recortado = [], 0.0
    for a in candidatas:
        if recortado >= deficit:
            break
        elegidas.append({"nombre": a.nombre, "horas": a.horas_estimadas})
        recortado += a.horas_estimadas
    return elegidas


def validar(plan: PlanEntrada, fecha_inicio: date, modo: Literal["nuevo", "replanificacion"] = "nuevo") -> Resultado:
    errores: list[Hallazgo] = []
    advertencias: list[Hallazgo] = []
    disp, tope = plan.disponibilidad, plan.max_horas_dia
    vigentes = [a for a in plan.actividades if a.fecha_limite >= fecha_inicio]

    if tope is not None and plan.duracion_sesion / 60 > tope:  # R3a
        errores.append(Hallazgo(
            codigo="R3A_SESION_MAYOR_MAXIMO",
            mensaje=f"La sesión máxima ({int(plan.duracion_sesion)} min) supera el máximo diario ({tope:g} h)",
            datos={"duracion_sesion_min": int(plan.duracion_sesion), "max_horas_dia": tope}))

    for a in plan.actividades:  # R4
        if a.fecha_limite < fecha_inicio:
            errores.append(Hallazgo(
                codigo="R4_FECHA_PASADA",
                mensaje=f"'{a.nombre}' vence el {a.fecha_limite}, antes del inicio del plan ({fecha_inicio})",
                datos={"actividad": a.nombre, "fecha_limite": a.fecha_limite.isoformat()}))

    pasados = sorted(d.dia for d in disp if d.dia < fecha_inicio)  # R5
    if pasados:
        advertencias.append(Hallazgo(
            codigo="R5_DISPONIBILIDAD_PASADA",
            mensaje=f"Se ignoran {len(pasados)} día(s) de disponibilidad anteriores al inicio del plan",
            datos={"dias": [d.isoformat() for d in pasados]}))

    repetidos = sorted(n for n, c in Counter(a.nombre.casefold() for a in plan.actividades).items() if c > 1)  # R6
    if repetidos:
        advertencias.append(Hallazgo(
            codigo="R6_NOMBRE_DUPLICADO",
            mensaje=f"Hay actividades con el mismo nombre: {', '.join(repetidos)}",
            datos={"nombres": repetidos}))

    for d in sorted(disp, key=lambda d: d.dia):  # R3b
        if d.dia >= fecha_inicio and d.franjas and capacidad_dia(d) == 0:
            advertencias.append(Hallazgo(
                codigo="R3B_DIA_SIN_CAPACIDAD",
                mensaje=f"El {d.dia} no aporta horas: todas sus franjas duran menos de 30 min",
                datos={"dia": d.dia.isoformat()}))

    for a in vigentes:  # R2
        n = dias_con_capacidad(disp, tope, fecha_inicio, a.fecha_limite)
        if n < VENTANA_MINIMA_DIAS:
            hallazgo = Hallazgo(
                codigo="R2_VENTANA_MINIMA",
                mensaje=f"'{a.nombre}' tiene {n} día(s) con disponibilidad antes de la entrega; el mínimo es {VENTANA_MINIMA_DIAS}",
                datos={"actividad": a.nombre, "dias_con_disponibilidad": n, "minimo": VENTANA_MINIMA_DIAS})
            (errores if modo == "nuevo" else advertencias).append(hallazgo)

    for limite in sorted({a.fecha_limite for a in vigentes}):  # R1 (cubre R7)
        vencen = [a for a in vigentes if a.fecha_limite <= limite]
        carga = sum(a.horas_estimadas for a in vencen)
        capacidad = capacidad_entre(disp, tope, fecha_inicio, limite)
        deficit = carga - (capacidad + MARGEN_HORAS)
        if deficit > 1e-9:
            errores.append(Hallazgo(
                codigo="R1_CAPACIDAD_INSUFICIENTE",
                mensaje=f"Hasta el {limite} hay {carga:g} h de trabajo y solo {capacidad:g} h disponibles (faltan {deficit:g} h)",
                datos={"fecha_critica": limite.isoformat(), "horas_requeridas": carga, "horas_disponibles": capacidad,
                       "deficit_horas": round(deficit, 4), "sugerencia_recorte": _sugerir_recorte(vencen, deficit)}))

    estado = Estado.INVIABLE if errores else Estado.CON_ADVERTENCIAS if advertencias else Estado.FACTIBLE
    return Resultado(estado=estado, errores=errores, advertencias=advertencias)

## 4.1 Un ejemplo completo

Tres actividades que vencen el mismo día. Cada una cabe sola en el tiempo disponible, pero juntas no: es el caso que la revisión acumulada detecta. El resultado trae la fecha crítica, el déficit y qué recortar (primero la de prioridad baja).

In [6]:
from datetime import timedelta

INICIO = date(2026, 10, 25)  # domingo


def f(n):
    return (INICIO + timedelta(days=n)).isoformat()


def act(nombre, n_limite, horas, prioridad="media", dificultad="media"):
    return {"nombre": nombre, "prioridad": prioridad, "dificultad": dificultad, "fecha_limite": f(n_limite), "horas_estimadas": horas}


def dias(desde, hasta, pares=(("18:00", "21:00"),)):
    return [{"dia": f(n), "franjas": franjas(*pares)} for n in range(desde, hasta)]


def plan(actividades, disponibilidad, sesion=60, max_horas_dia=None):
    return {"actividades": actividades, "disponibilidad": disponibilidad, "duracion_sesion": sesion, "max_horas_dia": max_horas_dia}


ejemplo = plan([act("Taller", 3, 4, "alta"), act("Lectura", 3, 3, "baja"), act("Informe", 3, 3, "media")], dias(0, 10))
r = validar(PlanEntrada(**ejemplo), INICIO)
print(r.estado.value)
print(r.model_dump_json(indent=2))

inviable
{
  "estado": "inviable",
  "errores": [
    {
      "codigo": "R1_CAPACIDAD_INSUFICIENTE",
      "mensaje": "Hasta el 2026-10-28 hay 10 h de trabajo y solo 9 h disponibles (faltan 1 h)",
      "datos": {
        "fecha_critica": "2026-10-28",
        "horas_requeridas": 10.0,
        "horas_disponibles": 9.0,
        "deficit_horas": 1.0,
        "sugerencia_recorte": [
          {
            "nombre": "Lectura",
            "horas": 3.0
          }
        ]
      }
    }
  ],
  "advertencias": []
}


**Observaciones**

1. Las tres vencen el miércoles (n=3). Los días de trabajo son domingo, lunes y martes, con 3 h cada uno: 9 h en total para 10 h de trabajo, faltan 1 h.
2. La sugerencia propone recortar `Lectura` (prioridad baja) y con eso cubre el déficit. Las otras dos actividades no se tocan.
3. `datos` trae todo lo necesario para que el LLM redactor explique el problema sin inventar cifras.

# PASO 5. CASOS DE PRUEBA

Cada caso guarda **entrada, salida esperada y salida lograda**, y esta tabla alimenta la sección 7 del informe. Los casos son datos (`CASOS`), así que después se reutilizan con `pytest.mark.parametrize` sin reescribirlos.

Todos usan `INICIO` = domingo 25 de octubre de 2026 y, salvo que se diga otra cosa, 3 h de disponibilidad por día (18:00-21:00) y sesión de 60 min. `n` en la columna de entrada es el día respecto al inicio (0 = domingo).

In [7]:
def esp(estado, errores=(), advertencias=()):
    return {"estado": estado, "errores": sorted(errores), "advertencias": sorted(advertencias)}


R1, R2, R3A, R3B = "R1_CAPACIDAD_INSUFICIENTE", "R2_VENTANA_MINIMA", "R3A_SESION_MAYOR_MAXIMO", "R3B_DIA_SIN_CAPACIDAD"
R4, R5, R6 = "R4_FECHA_PASADA", "R5_DISPONIBILIDAD_PASADA", "R6_NOMBRE_DUPLICADO"

CASOS = [
    {"nombre": "Plan factible holgado", "resumen": "2 actividades (7 h), 30 h libres",
     "entrada": plan([act("Taller", 7, 4), act("Informe", 10, 3)], dias(0, 10)), "esperado": esp("factible")},
    {"nombre": "Total inviable", "resumen": "40 h de trabajo, 15 h libres antes de la entrega",
     "entrada": plan([act("Tesis", 5, 40)], dias(0, 10)), "esperado": esp("inviable", [R1])},
    {"nombre": "Total viable, entrega cercana inviable", "resumen": "14 h en total (caben en 30 h), pero 10 h vencen en 9 h libres",
     "entrada": plan([act("Taller A", 3, 5), act("Taller B", 3, 5), act("Proyecto", 10, 4)], dias(0, 10)), "esperado": esp("inviable", [R1])},
    {"nombre": "Plan justo al 100 %", "resumen": "9 h de trabajo, 9 h libres (margen 0)",
     "entrada": plan([act("Examen", 3, 9)], dias(0, 5)), "esperado": esp("factible")},
    {"nombre": "Ventana de 2 días (R2, nuevo)", "resumen": "entrega en n=2, solo 2 días de trabajo",
     "entrada": plan([act("Quiz", 2, 2)], dias(0, 5)), "esperado": esp("inviable", [R2])},
    {"nombre": "Ventana de 2 días (R2, replanificación)", "resumen": "igual al anterior, modo replanificación", "modo": "replanificacion",
     "entrada": plan([act("Quiz", 2, 2)], dias(0, 5)), "esperado": esp("factible_con_advertencias", [], [R2])},
    {"nombre": "Ventana con un día sin disponibilidad", "resumen": "entrega en n=4, disponibilidad solo en n=0 y n=2",
     "entrada": plan([act("Lectura", 4, 2)], dias(0, 1) + dias(2, 3) + dias(5, 7)), "esperado": esp("inviable", [R2])},
    {"nombre": "Sesión mayor que el máximo diario (R3a)", "resumen": "sesión máxima de 90 min, máximo diario de 1 h",
     "entrada": plan([act("Taller", 7, 2)], dias(0, 10), sesion=90, max_horas_dia=1), "esperado": esp("inviable", [R3A])},
    {"nombre": "Día con franja menor a 30 min (R3b)", "resumen": "el día n=3 solo tiene 18:00-18:20",
     "entrada": plan([act("Taller", 7, 4)], dias(0, 3) + dias(3, 4, (("18:00", "18:20"),)) + dias(4, 10)),
     "esperado": esp("factible_con_advertencias", [], [R3B])},
    {"nombre": "Sobrante de la franja aprovechado", "resumen": "franjas de 4 h con sesión máxima de 90 min; 12 h de trabajo en 3 días",
     "entrada": plan([act("Proyecto", 3, 12)], dias(0, 5, (("16:00", "20:00"),)), sesion=90), "esperado": esp("factible")},
    {"nombre": "Fecha límite en el pasado (R4)", "resumen": "una actividad vence en n=-2",
     "entrada": plan([act("Entrega vieja", -2, 2), act("Taller", 7, 4)], dias(0, 10)), "esperado": esp("inviable", [R4])},
    {"nombre": "Disponibilidad de días pasados (R5)", "resumen": "disponibilidad desde n=-2",
     "entrada": plan([act("Taller", 7, 4)], dias(-2, 10)), "esperado": esp("factible_con_advertencias", [], [R5])},
    {"nombre": "Nombres duplicados (R6)", "resumen": "'Parcial' y 'parcial ' (con espacio final)",
     "entrada": plan([act("Parcial", 7, 2), act("parcial ", 9, 2)], dias(0, 10)), "esperado": esp("factible_con_advertencias", [], [R6])},
    {"nombre": "Prioridad con mayúsculas", "resumen": "prioridad 'ALTA' y dificultad ' Baja'",
     "entrada": plan([act("Taller", 7, 2, "ALTA", " Baja")], dias(0, 10)), "esperado": esp("factible")},
    {"nombre": "Días repetidos", "resumen": "el día n=2 aparece dos veces",
     "entrada": plan([act("Taller", 7, 2)], dias(0, 3) + dias(2, 5)), "esperado": {"pydantic": "repetidos"}},
    {"nombre": "Franjas traslapadas", "resumen": "18:00-20:00 y 19:00-21:00 el mismo día",
     "entrada": plan([act("Taller", 7, 2)], [{"dia": f(0), "franjas": franjas(("18:00", "20:00"), ("19:00", "21:00"))}]),
     "esperado": {"pydantic": "traslapan"}},
    {"nombre": "Disponibilidad vacía", "resumen": "lista de disponibilidad sin días",
     "entrada": plan([act("Taller", 7, 2)], []), "esperado": {"pydantic": "too_short"}},
    {"nombre": "Actividad de 0 horas", "resumen": "horas_estimadas = 0",
     "entrada": plan([act("Taller", 7, 0)], dias(0, 10)), "esperado": {"pydantic": "greater_than"}},
    {"nombre": "Nombre solo espacios", "resumen": "nombre = '   '",
     "entrada": plan([act("   ", 7, 2)], dias(0, 10)), "esperado": {"pydantic": "string_too_short"}},
    {"nombre": "Sesión fuera del enum", "resumen": "duracion_sesion = 45",
     "entrada": plan([act("Taller", 7, 2)], dias(0, 10), sesion=45), "esperado": {"pydantic": "enum"}},
]


def ejecutar(caso):
    try:
        p = PlanEntrada(**caso["entrada"])
    except ValidationError as e:
        return {"pydantic": "; ".join(f"{x['type']} ({x['msg']})" for x in e.errors())}
    r = validar(p, INICIO, caso.get("modo", "nuevo"))
    return esp(r.estado.value, [h.codigo for h in r.errores], [h.codigo for h in r.advertencias])


def coincide(esperado, lograda):
    if "pydantic" in esperado:
        return "pydantic" in lograda and esperado["pydantic"] in lograda["pydantic"]
    return esperado == lograda


def texto(salida):
    if "pydantic" in salida:
        return "rechaza: " + salida["pydantic"][:60]
    return " | ".join([salida["estado"]] + [f"E:{c.split('_')[0]}" for c in salida["errores"]] + [f"A:{c.split('_')[0]}" for c in salida["advertencias"]])


filas = []
for c in CASOS:
    lograda = ejecutar(c)
    filas.append({"Caso": c["nombre"], "Entrada": c["resumen"], "Esperada": texto(c["esperado"]),
                  "Lograda": texto(lograda), "OK": "sí" if coincide(c["esperado"], lograda) else "NO"})

import pandas as pd
pd.set_option("display.max_colwidth", 90)
tabla = pd.DataFrame(filas)
print(f"{(tabla['OK'] == 'sí').sum()} de {len(tabla)} casos coinciden con lo esperado")
tabla

20 de 20 casos coinciden con lo esperado


,Caso,Entrada,Esperada,Lograda,OK
0,Plan factible holgado,"2 actividades (7 h), 30 h libres",factible,factible,sí
1,Total inviable,"40 h de trabajo, 15 h libres antes de la entrega",inviable | E:R1,inviable | E:R1,sí
2,"Total viable, entrega cercana inviable","14 h en total (caben en 30 h), pero 10 h vencen en 9 h libres",inviable | E:R1,inviable | E:R1,sí
3,Plan justo al 100 %,"9 h de trabajo, 9 h libres (margen 0)",factible,factible,sí
4,"Ventana de 2 días (R2, nuevo)","entrega en n=2, solo 2 días de trabajo",inviable | E:R2,inviable | E:R2,sí
5,"Ventana de 2 días (R2, replanificación)","igual al anterior, modo replanificación",factible_con_advertencias | A:R2,factible_con_advertencias | A:R2,sí
6,Ventana con un día sin disponibilidad,"entrega en n=4, disponibilidad solo en n=0 y n=2",inviable | E:R2,inviable | E:R2,sí
7,Sesión mayor que el máximo diario (R3a),"sesión máxima de 90 min, máximo diario de 1 h",inviable | E:R3A,inviable | E:R3A,sí
8,Día con franja menor a 30 min (R3b),el día n=3 solo tiene 18:00-18:20,factible_con_advertencias | A:R3B,factible_con_advertencias | A:R3B,sí
9,Sobrante de la franja aprovechado,franjas de 4 h con sesión máxima de 90 min; 12 h de trabajo en 3 días,factible,factible,sí


In [8]:
assert (tabla["OK"] == "sí").all(), tabla[tabla["OK"] == "NO"]

**Observaciones**

1. Los casos de rechazo de Pydantic (últimos cinco) nunca llegan al validador de coherencia; los demás sí.
2. Dos casos revisan decisiones que se tomaron en este notebook: el sobrante de franja que se aprovecha y el plan justo al 100 %.
3. R7 no tiene caso propio porque lo cubre R1: con capacidad cero, cualquier actividad con horas positivas falla en R1.

# PASO 6. MIGRACIÓN A ARCHIVOS `.py`

| Celdas del notebook | Archivo | Imports que se agregan |
|---|---|---|
| Paso 2, celda `models.py` | `studyflow/models.py` | ninguno |
| Paso 3, celda `disponibilidad.py` | `studyflow/disponibilidad.py` | `from models import DisponibilidadDia` |
| Paso 4, celda `validador.py` | `studyflow/validador.py` | `from models import Nivel, PlanEntrada` y `from disponibilidad import capacidad_dia, capacidad_entre, dias_con_capacidad` |
| Pasos 2.1, 3.1 y 5 | `tests/` | los helpers (`act`, `dias`, `plan`, `dia`) van a `conftest.py`; `CASOS` se reutiliza con `pytest.mark.parametrize` |

Cuando la lógica esté en `.py`, el notebook importa desde los módulos y deja de definirla.

## Puntos para tener en cuenta en las otras capas

* **MLP:** las entradas 5 y 6 deben calcularse con `capacidad_entre` (misma función que el validador). El dataset se entrenó con plazos de **3 a 30 días** y ventanas de **6 a 240 h**; un plazo más largo queda fuera de ese rango y hay que limitarlo en la interfaz o recortarlo al escalar.
* **Capa de asignación:** parte cada franja en sesiones del máximo y deja la última más corta (mínimo 30 min); dentro del día llena primero la franja más temprana; los 3 tramos del reparto se aplican sobre los **días con capacidad**, no sobre los días del calendario.
* **Replanificación:** las actividades con menos de 3 días se asignan por prioridad, sin perfil de reparto (se define con la capa de asignación).
* **R3a:** hoy es un error que la sesión máxima supere el máximo diario. Como la duración ahora es un máximo, se podría convertir en advertencia; queda pendiente de decidir.